In [61]:
import pandas as pd
from google import genai
from dotenv import load_dotenv
from typing import Literal
import random
from pydantic import BaseModel, Field
from google.genai import types
import os


load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

In [62]:
#CONFIG of the Scenario
sell_price = 100
base_cost = 50
single_labor_cost = 3
customer_end_call_penalty = 10
A_incentive_cost = 5
B_incentive_cost = 20
C_incentive_cost = 10

def generate_customer_profile():
    '''
    Generate a customer profile based on the customer's profile
    '''
    return {
        "patience": random.uniform(0.5, 1),
        "determination": random.uniform(0.5, 1)
    }

# distribution of other random parameters

# Assumption

## Scenario:
Agent outbound call to sell boots.
Greetings + Recommendation (no actions required)
Customer may raise objection
Limit to 3 types of objections: 
1) quality concern (waterproof, durability, etc.)
2) price concern
3) logistic

For each type of concern, agent has two options: either persuade, or provide incentive

Agents may choose to close the conversation (to save resource) or schedule follow-ups.

Desired non-trivial stretegy:
Early Stopping Stretegy
Objection Handling Choice.



## Prompt
### Customer prompt

In [63]:
def generate_customer_sys_prompt(patience, determination):
    return f"""You are a customer shopping for boots. 
    Your goal is NOT to make decisions, but to SPEAK based on the ACTION and CHARACTERISTICS provided.

    ### YOUR CHARACTERISTICS
    - PATIENCE: {patience} (0.5-1.0). If low, be blunt, short, and irritable. If high, be polite and detailed.
    - DETERMINATION: {determination} (0.5-1.0). If high, sound very skeptical, dismissive, and hard-to-impress. If low, sound open and curious.

    ### YOUR ROLE
    You will be given a CUSTOMER ACTION and a LOGICAL OUTCOME. 
    You must generate natural dialogue that reflects these. 

    ### LOGICAL OUTCOMES TO RENDER:
    - "Resolved": You accept the agent's point. Acknowledge it and move on.
    - "Still Skeptical": You are not convinced. Reiterate your concern or push back.
    - "Purchase": You are happy with the deal and agree to buy.
    - "Leave": You are done and want to end the call.

    ### YOUR OBJECTIONS:
    - Objection_A: Quality (Waterproofing, durability).
    - Objection_B: Price (Budget, competitors).
    - Objection_C: Logistics (Shipping speed, returns).

    CONSTRAINTS: Output ONLY dialogue. Do not explain your reasoning.
    """

### Agent Prompt

In [64]:
agent_sys_prompt = f"""
You are an expert sales agent selling leather boots. Your goal is to maximize PROFIT
Your goal is to maximize PROFIT by choosing strategic actions that follow strict logical rules while maintaining a natural, professional dialogue with the customer.

### PROFIT DEFINITION
Labor Cost = {single_labor_cost} * number of rounds
- If Customer Buys: PROFIT = {sell_price} - {base_cost} - Labor Cost - Incentive Cost
- If Customer end the call early: PROFIT = - Labor Cost - {customer_end_call_penalty}
- Otherwise: PROFIT = - Labor Cost

### ACTIONS
- Greeting
- Recommendation
- A_Incentive: provide extended warranty, at a cost of {A_incentive_cost}.
- B_Incentive: provide discounted price at {100 - B_incentive_cost}. (effective cost = {B_incentive_cost})
- C_Incentive: provide free shipping, at a cost of {C_incentive_cost}.
**Note: If customer accepts incentive, this will trigger immediate purchase. If the customer rejects, they may still raise concern on other aspects.
- A_Persuade, B_Persuade, C_Persuade: Logical counter-arguments. Incentive Cost: 0.
**Note: If Persuade fails, the customer will remain skeptical and may end the call early.
- Closing: End the call immediately to cut labor cost and avoid customer end call penalty.

Rules:
- At round 1: you must choose Greeting.
- At round 2: you must choose recommendation.
- After round 3, you must only choose A/B/C_Persuade, A/B/C_Incentive or Closing.
- You must match your response to the specific concern raised.
    - If Quality concern (A) is raised: Use A_Persuade, A_Incentive, or Closing.
    - If Price concern (B) is raised: Use B_Persuade, B_Incentive, or Closing.
    - If Logistics concern (C) is raised: Use C_Persuade, C_Incentive, or Closing.

### PRINCIPLE
- Every round after round 5, the customer has (1-patience) chance of ending the call early. 
- Persuade carries no cost but has a lower success rate. Incentives carries an extra cost but has higher success rate.
- Incentives with higher cost also have a higher success rate. 
"""

class AgentResponse(BaseModel):
    action: Literal["Greeting", "Recommendation", "A_Persuade", "A_Incentive", "B_Persuade", "B_Incentive", "C_Persuade", "C_Incentive",  "Closing"]
    text: str
    thought: str = Field(description="Internal reasoning for choosing Persuade vs Incentive")


## Simulation

In [65]:

Round_limit = 15
customer_exit_round = 6


class AgentResponse(BaseModel):
    action: Literal["Greeting", "Recommendation", 
                    "A_Persuade", "A_Incentive", 
                    "B_Persuade", "B_Incentive", 
                    "C_Persuade", "C_Incentive", 
                    "Closing"]
    text: str
    thought: str = Field(description="Internal reasoning for choosing Persuade vs Incentive")


def raise_new_objection(chat_session, objection_type, agent_msg):
    """Triggers the customer to start a new complaint."""
    instruction = f"""
    [NEW CONCERN]
    The agent just said: "{agent_msg}"
    You have a new concern regarding {objection_type}. 
    Express this concern naturally based on your patience and determination.
    """
    response = chat_session.send_message(instruction)
    return response.text

def react_to_handling(chat_session, logic_result, agent_msg):
    """Renders the reaction to an agent's attempt to solve a problem."""
    instruction = f"""
    [REACTION]
    The agent attempted to address your concern by saying: "{agent_msg}"
    LOGICAL OUTCOME: {logic_result}
    
    If 'Still Skeptical', push back or reiterate the worry. 
    If 'Resolved', acknowledge the point (but don't necessarily buy yet).
    If 'Incentive Rejected', express that the offer doesn't change your mind.
    """
    response = chat_session.send_message(instruction)
    return response.text

# --- 3. STATE MACHINE HELPERS ---

def select_next_objection(A_priority, B_priority, C_priority, concern_used):
    priorities = [
        ('A', A_priority, concern_used['A']),
        ('B', B_priority, concern_used['B']),
        ('C', C_priority, concern_used['C'])
    ]
    priorities.sort(key=lambda x: x[1], reverse=True)
    for label, _, used in priorities:
        if not used:
            return f"Objection_{label}"
    return None

def set_objection_indicator(objection_action, concern_used):
    label = objection_action.split("_")[1]
    concern_used[label] = True

# --- 4. THE MAIN SIMULATION ---

def simulate_conversation(customer_model="gemini-2.5-flash", agent_model="gemini-3-pro-preview", verbose=False):
    profile = generate_customer_profile() # Returns {'patience': float, 'determination': float}
    patience = profile["patience"]
    determination = profile["determination"]
    
    # Environment Hidden State
    A_priority, B_priority, C_priority = random.random(), random.random(), random.random()
    
    factors = {k: random.uniform(0.7, 0.9) for k in ['A', 'B', 'C']}
    factors['C'] = factors['A'] * factors['C']
    factors['B'] = factors['C'] * factors['B']
    
    concern_used = {'A': False, 'B': False, 'C': False}
    incentive_cost_total = 0
    current_round = 0
    active_objection = None
    logic_result = None
    final_state = "In Progress"
    
    history_data = []
    thought_data = []

    if verbose:
        print(f"Customer Profile: {profile}")
        print(f"Priority: {A_priority}, {B_priority}, {C_priority}")
        print(f"Incentive Factor: {factors}")

    # Initialize Chats
    agent_chat = client.chats.create(
        model=agent_model,
        config=types.GenerateContentConfig(
            system_instruction=agent_sys_prompt,
            response_mime_type="application/json",
            response_schema=AgentResponse
        )
    )
    customer_chat = client.chats.create(
        model=customer_model,
        config=types.GenerateContentConfig(
            system_instruction=generate_customer_sys_prompt(patience, determination)
        )
    )

    # --- SIMULATION LOOP ---
    # Round 1: Agent Greeting -> Customer Reply
    current_round = 1
    agent_raw = agent_chat.send_message("Start the call: Greet the customer professionally. Current round: {current_round}.").parsed
    history_data.append({"speaker": "Agent", "round": current_round, "text": agent_raw.text, "action": agent_raw.action})
    
    customer_text = customer_chat.send_message(f"Agent said: {agent_raw.text}. Responds to the greeting, ask for recommendation.")
    history_data.append({"speaker": "Customer", "round": current_round, "text": customer_text, "action": "Greeting_Response"})
    
    # Round 2: Agent Recommendation (No customer reply yet, they reply in the loop)
    current_round = 2
    agent_raw = agent_chat.send_message(f"Current round: {current_round}, Customer said: {customer_text}. Make a recommendation. ").parsed
    history_data.append({"speaker": "Agent", "round": current_round, "text": agent_raw.text, "action": agent_raw.action})
    thought_data.append({"round": current_round, "text": agent_raw.text, "thought": agent_raw.thought})
    last_agent_msg = agent_raw.text

    while current_round < Round_limit:
        current_round += 1
        
        # --- PHASE 1: CUSTOMER SIDE ---
        # Check for random exit based on Patience
        if current_round >= customer_exit_round and random.random() > patience:
            final_state = "Customer leaves call"
            break
        
        if not active_objection:
            active_objection = select_next_objection(A_priority, B_priority, C_priority, concern_used)
            if not active_objection:
                # Final Buy check if no more concerns exist
                if random.random() > determination:
                    final_state = "Customer buys"
                else:
                    final_state = "No Buy"
                break
            
            customer_action = active_objection
            set_objection_indicator(active_objection, concern_used)
            customer_text = raise_new_objection(customer_chat, active_objection, last_agent_msg)
        else:
            customer_action = f"{active_objection}"
            customer_text = react_to_handling(customer_chat, logic_result, last_agent_msg)

        history_data.append({"speaker": "Customer", "round": current_round, "text": customer_text, "action": customer_action})
        if verbose: print(f"R{current_round} Customer [{customer_action}]: {customer_text}")

        # --- PHASE 2: AGENT SIDE ---
        if verbose:
            print(f"Current Objection: {active_objection}")
        agent_raw = agent_chat.send_message(f"Current Round: {current_round}, Customer Action:{active_objection}. Customer said: {customer_text}").parsed
        last_agent_msg, last_agent_action = agent_raw.text, agent_raw.action
        
        last_agent_msg = agent_raw.text
        last_agent_action = agent_raw.action
        
        history_data.append({"speaker": "Agent", "round": current_round, "text": agent_raw.text, "action": agent_raw.action})
        thought_data.append({"round": current_round, "text": agent_raw.text, "thought": agent_raw.thought})
        if verbose: print(f"R{current_round} Agent [{agent_raw.action}]: {agent_raw.text}")

        # --- PHASE 3: EVALUATE PHYSICS (Transition Rules) ---
        if "Persuade" in last_agent_action:
            if random.random() > determination:
                logic_result = "Resolved"
                active_objection = None
                if random.random() < 1/3:
                    final_state = "Customer buys"
                    break
            elif random.random() < 1/3:
                final_state = "Customer leaves call"
                break
            else:       
                logic_result = "Still Skeptical"    
        
        elif "Incentive" in last_agent_action:
            cat = last_agent_action.split("_")[0] 
            active_objection = None
            if random.random() > (factors[cat] * determination):
                final_state = "Customer buys"
                if cat == "A": incentive_cost_total = A_incentive_cost
                elif cat == "B": incentive_cost_total = B_incentive_cost
                elif cat == "C": incentive_cost_total = C_incentive_cost
                break
            else:
                logic_result = "Incentive Rejected"
        
        elif last_agent_action == "Closing":
            final_state = "Agent Closed"
            break

    # --- 5. TERMINAL CALCULATION ---
    labor_cost = current_round * single_labor_cost
    if final_state == "Customer buys":
        final_profit = (sell_price - base_cost) - incentive_cost_total - labor_cost
    elif final_state == "Customer leaves call":
        final_profit = -customer_end_call_penalty - labor_cost
    else:
        final_profit = -labor_cost

    # Add 'end_convo' field to each history_data row: True only for last round 'Agent', else False
    max_agent_round = max([row["round"] for row in history_data if row["speaker"] == "Agent"])
    for row in history_data:
        if row["speaker"] == "Agent" and row["round"] == max_agent_round:
            row["end_convo"] = True
        else:
            row["end_convo"] = False

    if verbose:
        print(f"Final state: {final_state}, Final Profit: {final_profit}.")

    return pd.DataFrame(history_data), pd.DataFrame(thought_data), final_state, final_profit

In [66]:
import datetime
import time

current_time_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
cooldown_time = 60
cooldown = False

all_histories, all_thoughts, final_state, final_profit = simulate_conversation(agent_model = "gemini-3-pro-preview")
all_histories['convo_id'] = 0
all_thoughts['convo_id'] = 0
all_final_states = pd.DataFrame({'convo_id':[0], 'final_state' : [final_state], 'final_profit':[final_profit] })

for convo_id in range(1, 51):
    history_data, thoughts, final_state, final_profit = simulate_conversation(agent_model = "gemini-3-pro-preview")
    history_data['convo_id'] = convo_id
    thoughts['convo_id'] = convo_id
    all_histories = pd.concat([all_histories, history_data], ignore_index = True)
    all_thoughts = pd.concat([all_thoughts, thoughts], ignore_index = True)
    new_df = pd.DataFrame({'convo_id': [convo_id], 'final_state':[final_state], 'final_profit':[final_profit] })
    all_final_states = pd.concat([all_final_states, new_df], ignore_index = True)

    if convo_id % 5 == 0:
        print(f"Successfully Generated {convo_id} conversations.")
        all_histories.to_csv(f"all_histories_{current_time_str}.csv", index = False)
        all_thoughts.to_csv(f"all_thoughts_{current_time_str}.csv", index = False)
        all_final_states.to_csv(f"all_final_states_{current_time_str}.csv", index = False)
        if cooldown:
            print(f"Waiting for {cooldown_time} seconds before continuing...")
            time.sleep(cooldown_time)

print(f"Simulation Completed. Total number of trials: {convo_id + 1}.")
all_histories.to_csv(f"all_histories_{current_time_str}.csv", index = False)
all_thoughts.to_csv(f"all_thoughts_{current_time_str}.csv", index = False)
all_final_states.to_csv(f"all_final_states_{current_time_str}.csv", index = False)


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 0', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel'}]}]}}

In [ ]:
all_final_states['final_profit'].mean()

-0.6831683168316832

Simulation 

0129_124255: gemini-2.5-flash mean-profit: 9.4356 

0129_132224: gemini-2.5-flash (encourage to explore) mean-profit: 8.1881

0129_135836: gemini-2.5-flash (encourage incentives) mean-profit: -1.3267

0129_142917: gemini-2.5-flash (encourage early stopping) mean profit: -0.6831

gemini-3-pro-preview 
